## Ejercicio 0 – Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocesamiento
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score

# Modelos de clasificación
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Modelos de regresión
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

print('¡Librerías importadas correctamente!')

## #1 – Explicación del dataset y carga de datos

In [ ]:
ruta = r'D:\The bridge\DS-Online-Hugo-Sanz\04_Machine_Learning\Sprint_12\Unidad_02_ML_Supervisado_Repaso\03_Practica_Obligatoria\data\wines_dataset.csv'
df = pd.read_csv(ruta, sep='|')

print('Filas y columnas:', df.shape)
df.head()

In [ ]:
# Información general: tipos de datos y nulos
df.info()

In [ ]:
# Estadísticas descriptivas
df.describe()

### Variables del dataset:

| Variable | Tipo | Descripción |
|---|---|---|
| fixed acidity | Numérica | Ácidos fijos del vino |
| volatile acidity | Numérica | Ácido acético (niveles altos → sabor a vinagre) |
| citric acid | Numérica | Añade frescura y sabor |
| residual sugar | Numérica | Azúcar restante tras la fermentación |
| chlorides | Numérica | Cantidad de sal |
| free sulfur dioxide | Numérica | SO2 libre (previene microbios y oxidación) |
| total sulfur dioxide | Numérica | SO2 total (libre + ligado) |
| density | Numérica | Densidad del vino |
| pH | Numérica | Acidez/basicidad (0–14) |
| sulphates | Numérica | Aditivo antimicrobiano y antioxidante |
| alcohol | Numérica | Porcentaje de alcohol |
| quality | Numérica (discreta) | Puntuación sensorial (0–10) → **target clasificación** |
| class | Categórica | Tinto o blanco → **feature** |

**Variables target:**
- **Clasificación:** `quality` (clases de 3 a 9)
- **Regresión:** `alcohol` (grado alcohólico, valor continuo)

El enunciado indica que el dataset **no tiene valores nulos** ni requiere limpieza.

### Distribución de los targets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target clasificación: quality
calidad = df['quality'].value_counts().sort_index()
calidad.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Distribución de Quality (clasificación)')
axes[0].set_xlabel('Puntuación de calidad')
axes[0].set_ylabel('Número de vinos')
axes[0].set_xticklabels(calidad.index, rotation=0)

# Target regresión: alcohol
axes[1].hist(df['alcohol'], bins=30, color='tomato', edgecolor='black')
axes[1].set_title('Distribución de Alcohol % (regresión)')
axes[1].set_xlabel('Grado alcohólico (%)')
axes[1].set_ylabel('Número de vinos')

plt.tight_layout()
plt.show()

### Assessment previo:

**Problema 1 – Clasificación (quality):**
- Las clases están **desbalanceadas**: la mayoría de vinos son de calidad 5, 6 o 7. Las clases extremas (3, 4, 8, 9) tienen muy pocas muestras.
- Esto dificultará que el modelo aprenda a predecir las clases minoritarias → usaremos `class_weight='balanced'` en los modelos que lo permitan.
- Métrica objetivo: **recall medio** (promedio del recall de todas las clases).

**Problema 2 – Regresión (alcohol):**
- El alcohol sigue una distribución ligeramente asimétrica a la derecha, con valores entre 8% y 15%.
- Métrica objetivo: **MAPE** (error porcentual medio absoluto), para equivocarse lo menos posible en términos relativos.

**Nota sobre `class`:** Es una variable categórica (tinto/blanco) que codificaremos numéricamente antes de modelar.

## #2 – Modelado para clasificación

**Objetivo:** Predecir la calidad del vino (`quality`) a partir de sus propiedades fisicoquímicas.

**Pasos:**
1. Preparar los datos (codificar `class`, separar X e y)
2. Dividir en train y test
3. Escalar
4. Comparar modelos con validación cruzada
5. Ajustar hiperparámetros del mejor modelo
6. Evaluar y analizar errores

### Paso 1 – Preparar los datos

In [ ]:
df_model = df.copy()

# Codificamos 'class': white=1, red=0
df_model['class'] = (df_model['class'] == 'white').astype(int)

# Para clasificación: X sin quality ni alcohol, y = quality
X_clf = df_model.drop(columns=['quality', 'alcohol'])
y_clf = df_model['quality']

print('Clases a predecir:', sorted(y_clf.unique()))
print('Distribución:')
print(y_clf.value_counts().sort_index())

### Paso 2 – Train/test split

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print('Train:', X_train_c.shape, '| Test:', X_test_c.shape)

### Paso 3 – Escalar

In [ ]:
scaler_c = StandardScaler()
X_train_c_sc = scaler_c.fit_transform(X_train_c)
X_test_c_sc  = scaler_c.transform(X_test_c)
print('Escalado listo.')

### Paso 4 – Comparar modelos con validación cruzada

Probamos:
- **KNN con K=3** (baseline)
- **KNN con K=7** (baseline con otro K)
- **Árbol de Decisión**
- **Random Forest** (con `class_weight='balanced'` para compensar el desbalanceo)

In [ ]:
cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modelos_clf = {
    'KNN (K=3)':           KNeighborsClassifier(n_neighbors=3),
    'KNN (K=7)':           KNeighborsClassifier(n_neighbors=7),
    'Árbol de Decisión':   DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
}

resultados_clf = {}
for nombre, modelo in modelos_clf.items():
    scores = cross_val_score(modelo, X_train_c_sc, y_train_c, cv=cv_c, scoring='recall_macro')
    resultados_clf[nombre] = scores
    print(f'{nombre:25s}: Recall medio = {scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
# Boxplot comparativo
plt.figure(figsize=(9, 5))
plt.boxplot(resultados_clf.values(), labels=resultados_clf.keys())
plt.title('Comparación de modelos – Recall macro (Clasificación)')
plt.ylabel('Recall macro')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Paso 5 – Optimización de hiperparámetros: Random Forest

El Random Forest suele ser el mejor modelo en este tipo de problemas. Ajustamos sus parámetros con GridSearchCV.

In [ ]:
param_grid_clf = {
    'n_estimators': [100, 200],
    'max_depth':    [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_clf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid=param_grid_clf,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='recall_macro',
    n_jobs=-1,
    verbose=1
)

grid_clf.fit(X_train_c_sc, y_train_c)

print('Mejores hiperparámetros:', grid_clf.best_params_)
print('Mejor recall en validación:', round(grid_clf.best_score_, 4))

### Paso 6 – Evaluación final en test y análisis de errores

In [ ]:
mejor_clf = grid_clf.best_estimator_
y_pred_c = mejor_clf.predict(X_test_c_sc)

print('=== Classification Report ===')
print(classification_report(y_test_c, y_pred_c))

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test_c, y_pred_c)
ConfusionMatrixDisplay(cm, display_labels=sorted(y_clf.unique())).plot(ax=ax, cmap='Blues')
ax.set_title('Matriz de Confusión – Random Forest (Clasificación)')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables
importancias_c = pd.Series(
    mejor_clf.feature_importances_,
    index=X_clf.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importancias_c.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Importancia de variables – Random Forest (Clasificación)')
plt.ylabel('Importancia')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Análisis de errores (Clasificación):

- La **diagonal** de la matriz de confusión muestra los aciertos. Las celdas fuera de diagonal son errores.
- Los errores más frecuentes ocurren entre **clases adyacentes** (p.ej. predecir un 6 cuando era un 5 o un 7), lo cual es comprensible porque la frontera entre calidades contiguas es difusa.
- Las clases extremas (3, 4, 8, 9) tienen **muy pocos datos** y son las más difíciles de predecir correctamente. El `class_weight='balanced'` ayuda pero no resuelve del todo el problema.
- Las variables más importantes suelen ser **alcohol**, **volatile acidity** y **sulphates**.

### Posibles mejoras:
- **Agrupar clases**: reducir las 7 clases a 3 (baja: 3-4, media: 5-6, alta: 7-9) simplificaría el problema y mejoraría el recall.
- Probar **GradientBoosting** o **XGBoost** que suelen rendir mejor en clasificación multiclase.
- Aumentar el número de muestras de las clases minoritarias con técnicas de oversampling adaptadas a multiclase.

## #3 – Modelado para regresión

**Objetivo:** Predecir el **grado alcohólico** (`alcohol`) a partir del resto de propiedades (incluida `quality`).

**Métrica:** MAPE (Mean Absolute Percentage Error) — minimiza el error porcentual.

Reutilizamos la codificación de `class` que ya hicimos antes.

### Paso 1 – Preparar los datos para regresión

In [ ]:
# Para regresión: X sin alcohol (pero SÍ incluye quality), y = alcohol
X_reg = df_model.drop(columns=['alcohol'])
y_reg = df_model['alcohol']

print('Features:', list(X_reg.columns))
print('Target: alcohol  |  Media:', round(y_reg.mean(), 2), ' | Std:', round(y_reg.std(), 2))

### Paso 2 – Train/test split y escalado

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_r = StandardScaler()
X_train_r_sc = scaler_r.fit_transform(X_train_r)
X_test_r_sc  = scaler_r.transform(X_test_r)

print('Train:', X_train_r.shape, '| Test:', X_test_r.shape)

### Paso 3 – Comparar modelos de regresión con validación cruzada

Probamos:
- **Regresión Lineal** (baseline sencillo)
- **KNN Regressor**
- **Random Forest Regressor**
- **Gradient Boosting Regressor**

Usamos **neg_mean_absolute_percentage_error** como scoring (negativo porque sklearn siempre maximiza; cuanto más cercano a 0 mejor).

In [ ]:
cv_r = KFold(n_splits=5, shuffle=True, random_state=42)

modelos_reg = {
    'Regresión Lineal':        LinearRegression(),
    'KNN Regressor':           KNeighborsRegressor(n_neighbors=5),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':       GradientBoostingRegressor(n_estimators=100, random_state=42),
}

resultados_reg = {}
for nombre, modelo in modelos_reg.items():
    scores = cross_val_score(
        modelo, X_train_r_sc, y_train_r,
        cv=cv_r, scoring='neg_mean_absolute_percentage_error'
    )
    mape_scores = -scores  # convertimos a positivo
    resultados_reg[nombre] = mape_scores
    print(f'{nombre:28s}: MAPE medio = {mape_scores.mean():.4f} (+/- {mape_scores.std():.4f})')

In [ ]:
# Boxplot comparativo (menos es mejor)
plt.figure(figsize=(10, 5))
plt.boxplot(resultados_reg.values(), labels=resultados_reg.keys())
plt.title('Comparación de modelos – MAPE (Regresión) · Menos es mejor')
plt.ylabel('MAPE')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Paso 4 – Optimización de hiperparámetros: Random Forest Regressor

El Random Forest o Gradient Boosting suelen ser los mejores modelos de regresión. Ajustamos Random Forest.

In [ ]:
param_grid_reg = {
    'n_estimators': [100, 200],
    'max_depth':    [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_reg = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_reg,
    cv=KFold(n_splits=3, shuffle=True, random_state=42),
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
    verbose=1
)

grid_reg.fit(X_train_r_sc, y_train_r)

print('Mejores hiperparámetros:', grid_reg.best_params_)
print('Mejor MAPE en validación:', round(-grid_reg.best_score_, 4))

### Paso 5 – Evaluación final y análisis de errores

In [ ]:
mejor_reg = grid_reg.best_estimator_
y_pred_r = mejor_reg.predict(X_test_r_sc)

mape  = mean_absolute_percentage_error(y_test_r, y_pred_r)
rmse  = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
r2    = r2_score(y_test_r, y_pred_r)

print(f'MAPE : {mape:.4f}  ({mape*100:.2f}% de error medio relativo)')
print(f'RMSE : {rmse:.4f}  (error medio en grados de alcohol)')
print(f'R²   : {r2:.4f}   (proporción de varianza explicada)')

In [ ]:
# Gráfico: valores reales vs predichos
plt.figure(figsize=(7, 6))
plt.scatter(y_test_r, y_pred_r, alpha=0.4, color='steelblue', edgecolors='none')
plt.plot([y_test_r.min(), y_test_r.max()],
         [y_test_r.min(), y_test_r.max()], 'r--', lw=2, label='Predicción perfecta')
plt.xlabel('Alcohol real (%)')
plt.ylabel('Alcohol predicho (%)')
plt.title('Valores reales vs predichos – Random Forest Regressor')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de los residuos (errores)
residuos = y_test_r.values - y_pred_r

plt.figure(figsize=(8, 4))
plt.hist(residuos, bins=40, color='tomato', edgecolor='black')
plt.axvline(0, color='black', linestyle='--', lw=2)
plt.xlabel('Error (real - predicho)')
plt.ylabel('Frecuencia')
plt.title('Distribución de residuos')
plt.tight_layout()
plt.show()

print(f'Media de los residuos:  {residuos.mean():.4f}  (idealmente cercana a 0)')
print(f'Std de los residuos:    {residuos.std():.4f}')

In [ ]:
# Importancia de variables en regresión
importancias_r = pd.Series(
    mejor_reg.feature_importances_,
    index=X_reg.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importancias_r.plot(kind='bar', color='tomato', edgecolor='black')
plt.title('Importancia de variables – Random Forest Regressor')
plt.ylabel('Importancia')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Análisis de errores (Regresión):

- El gráfico **real vs predicho** muestra qué tan cerca están los puntos de la línea diagonal roja. Cuanto más concentrados en la diagonal, mejor el modelo.
- La **distribución de residuos** debería ser simétrica y centrada en 0. Si hay sesgo (desplazamiento hacia un lado), el modelo tiende a sobre o infraestimar.
- Las variables más importantes para predecir el alcohol suelen ser **density** y **residual sugar** (a mayor azúcar y menor densidad, mayor alcohol).
- Un MAPE bajo (p.ej. < 5%) indica que el modelo comete pocos errores porcentuales, lo cual es bueno para las simulaciones de negocio.

### Posibles mejoras:
- Probar **Gradient Boosting** si da mejor MAPE en la comparación.
- Hacer **feature engineering**: por ejemplo, ratios entre variables (azúcar/densidad) que puedan mejorar la predicción.
- Analizar si los vinos tintos y blancos tienen comportamientos muy diferentes y considerar entrenar modelos separados para cada clase.